In [1]:
!pip3 install --upgrade --quiet  langchain langchain-community 	# core and extra packages
!pip3 install --upgrade --quiet  langchain-google-genai 		# Gemini lanfchain models
!pip3 install --upgrade --quiet  langchain-text-splitters 		
!pip3 install --upgrade --quiet  langchain-classic 				# load_evaluator
!pip3 install --upgrade --quiet  chromadb 						# vector database to look up the question embedding and the stored chinks embeddings
!pip3 install --upgrade --quiet  langchain-chroma 				# 
!pip3 install --upgrade --quiet  pypdf pandas streamlit			 
!pip3 install --upgrade --quiet  python-dotenv					# load_dotenv; reading env file


In [2]:
# IMPORT LANGCHAIN MODULES
from langchain_community.document_loaders import PyPDFLoader
	# reads PDF files and turns them into text langchain can work with

from langchain_text_splitters import RecursiveCharacterTextSplitter
	# cuts that text into smaller chunks (AI can't read huge docs in one go)

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
	# Gemini embeddings model and a chat model

from langchain_community.vectorstores import Chroma
	# a local database that stores your chunks in a searchable way

from langchain_core.runnables import RunnablePassthrough
	# plumbing - connects steps of your pipeline together

from langchain_core.prompts import ChatPromptTemplate
	# lets you write a template for how to instruct the AI

from pydantic import BaseModel, Field
	# used for defining structured/typed outputs - fairly advanced

from langchain_classic.evaluation import load_evaluator
	# used for embedding evaluation

from langchain_chroma import Chroma
	# used for loading vector storage

# OTHER PACKAGES
import os
	# interact with your operating system (e.g. read env variables)
import tempfile
	# create temporary files that auto-delete
import uuid
	# generate unique random IDs
import pandas as pd
	# spreadsheet-like data manipulation
import re
	# text pattern matching
from dotenv import load_dotenv
	# reads your API key from a .env file
from IPython.display import display
	# dosplays tables in a nice way

In [3]:
load_dotenv()
API_KEY = os.environ.get("GEMINI_API_KEY")

```
Using Gemini's free version
```

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=API_KEY
)

In [ ]:
# TEST
print(llm.model)
response = llm.invoke("hi")
print(response.content)

gemini-2.5-flash-lite


```
Load the documents
```

In [5]:
loader = PyPDFLoader("data/TIN_description_pdfs/austria-tin.pdf")
pages = loader.load()
pages

[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2016-08-18T14:02:51+02:00', 'author': 'MAGNIN Fedora', 'moddate': '2021-11-17T09:17:38+01:00', 'source': 'data/TIN_description_pdfs/austria-tin.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Jurisdiction’s name:  AUSTRIA \n \nInformation on Tax Identification Numbers \nSection I – TIN Description \n \nAustria issues TINs which are not reported on official documents of identification. The Local Tax \nOffices issue TINs to  the taxpayers having their residence in the area of competence of these Offices, \nwhen they ask the local office for a service. This means that a TIN can change when a taxpayer changes \nresidence. \nAdditional information on the mandatory issuance of Tax Identification Numbers (TINs) \nQuestion 1 – Does your jurisdiction automatically issue TINs to all residents for tax purposes? \nIndividuals No Entities Yes \n \nQuestion 2a – If you ans

```
Split the documents
```

In [10]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
											   chunk_overlap=50,
											   length_function=len, #how we count the characters
											   separators=["\n\n", 	#page break
						  									"\n",	#line break
															" "])
chunks = text_splitter.split_documents(pages)
chunks

[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2016-08-18T14:02:51+02:00', 'author': 'MAGNIN Fedora', 'moddate': '2021-11-17T09:17:38+01:00', 'source': 'data/TIN_description_pdfs/austria-tin.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Jurisdiction’s name:  AUSTRIA \n \nInformation on Tax Identification Numbers \nSection I – TIN Description \n \nAustria issues TINs which are not reported on official documents of identification. The Local Tax \nOffices issue TINs to  the taxpayers having their residence in the area of competence of these Offices, \nwhen they ask the local office for a service. This means that a TIN can change when a taxpayer changes \nresidence.'),
 Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2016-08-18T14:02:51+02:00', 'author': 'MAGNIN Fedora', 'moddate': '2021-11-17T09:17:38+01:00', 'source': 'data/TIN_description_pdfs/au

In [ ]:
get_embedding = GoogleGenerativeAIEmbeddings(
		model="gemini-embedding-001", google_api_key=API_KEY
	)
evaluator = load_evaluator(evaluator="embedding_distance",
						   embeddings=get_embedding)

In [ ]:
#TEST
test_1 = get_embedding.embed_query("TIN")
test_2 = get_embedding.embed_query("tax")

test_3 = evaluator.evaluate_strings(prediction="the dog chased the cat", reference="a puppy ran after a kitten")
test_4 = evaluator.evaluate_strings(prediction="the dog chased the cat", reference="the stock market crashed yesterday")
print(test_3)
print(test_4)

{'score': 0.0782906065960659}
{'score': 0.22745321449824485}


Chroma datbase

In [13]:
def create_vector_storage(chunks, embedding_function, storage_path):
	# Making sure there are no duplicates
	# Create a list of unique ids for each document based on the content
	ids = []
	for doc in chunks:		# doc is a temporary variable for iteration
		doc_page_content = doc.page_content				# doc is a LangChain document object containing plain text and metadata, so we need to access plain text
		seed = uuid.NAMESPACE_DNS						# seed for the mathematical formula (called SHA-1 hashing). This one is the conventional default
		myUniqueID = uuid.uuid5(seed, doc_page_content)	# uuid5 is deterministic: same chink always creates the same id
		ids.append(str(myUniqueID))
	# the above is identical to this "list comprehension"
	#		ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.page_content)) for doc in chunks]

	unique_ids = set()
		# an emtpy container. set is a list without duplicates
		# this will store all the uniques ids
	unique_chunks = []
		# this will store all the uniques chunks

	# Ensure that only unique docs with unique ids are kept
	for chunk, id in zip(chunks, ids):		# zip pairs the chunks with the ids.  
		if id not in unique_ids:       		# loops and adds if we dont have it yet
			unique_ids.add(id)
			unique_chunks.append(chunk) 
       
	# Creating the chroma database
	vector_storage = Chroma.from_documents(documents = unique_chunks,
										ids = list(unique_ids),
										embedding = embedding_function,
										persist_directory = storage_path)
	#vector_storage.persist() #does it automatically
	return vector_storage

In [ ]:
vector_storage = create_vector_storage(chunks=chunks,
									   embedding_function=get_embedding,
									   storage_path="vector_storage_chroma")
#load vector_storage
vector_storage = Chroma(persist_directory="vector_storage_chroma", embedding_function=get_embedding)

In [16]:
retriever = vector_storage.as_retriever(search_type="similarity") 	#uses cos(distance) to determine similarity
# similarity:	Finds the chunks whose vectors are closest to the query vector. Pure distance measurement.
# mmr: 			Maximum Marginal Relevance: If two chunks are very similar to each other, it picks one and skips the other in favour of something slightly less relevant but more different.
#				Useful when your PDF has repetitive content 
# similarity_score_threshold: Same as similarity but lets you set a minimum score — chunks below that threshold get rejected entirely even if they're the closest available.
#				Useful when you'd rather return nothing than return something irrelevant.
# extra parameter:  how many chunks get returned:
# search_kwargs={"k": 5}  # default is 4

relevant_chunks = retriever.invoke("What is the TIN structure for this country?")
print(relevant_chunks)


[Document(id='3e4dea33-780a-561c-b149-aafe053eac83', metadata={'creator': 'Microsoft® Word 2010', 'producer': 'Microsoft® Word 2010', 'source': 'data/TIN_description_pdfs/austria-tin.pdf', 'moddate': '2021-11-17T09:17:38+01:00', 'page_label': '1', 'page': 0, 'total_pages': 2, 'creationdate': '2016-08-18T14:02:51+02:00', 'author': 'MAGNIN Fedora'}, page_content='Question 2b – If you answered no to Question 1 with respect to Entities (as defined by the CRS), \ndescribe those instances where Entities are not being automatically issued a TIN. \nSection II – TIN Structure \n \nFormat Explanation Comment \n99-999/9999 9 digits The hyphen and the slash are \nnot mandatory in all cases (eg \nfor the purpose of IT \nprocessing they should be \nomitted) \n \n \nSection III – Where to find TINs?'), Document(id='cf6d0cd5-e949-5947-8e0a-9ba588344ec1', metadata={'author': 'MAGNIN Fedora', 'total_pages': 2, 'creator': 'Microsoft® Word 2010', 'moddate': '2021-11-17T09:17:38+01:00', 'producer': 'Micros

In [17]:
#Prompt instructions
PROMPT_TEMPLATE = """
Use the following pieces of retieved information only to answer the question
If you don't know the answer, say "I don't know", Don't make up anything.
If there is ambiguity, say "I am uncertain"API_KEY

{context}

Answer based on the above context: {question}
"""


In [ ]:
formatting = "\n\n---\n\n".join([doc.page_content for doc in relevant_chunks])

prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
promt = prompt_template.format(context=formatting,
							   question="How are the TINs structured for this country?")
print(promt)

In [ ]:
llm.invoke(promt)

In [20]:
#pydantic is a data validation lib for python. to specify the stucture of the answer

class structuredAnswer(BaseModel):
	answer: str = Field(description="Concise answer to the question")
	sources: str = Field(description="Full direct text chunk from the context used to answer the question")
	regex: str = Field(description="Create a regex that will be processed and compared to real life TINs, and need to be reliably determined if the TIN fufills the regex")

In [21]:
def format_docs(docs):
	return "\n\n---\n\n".join([doc.page_content for doc in docs])

rag_chain = (
			{"context": retriever | format_docs, "question":RunnablePassthrough()}
			| prompt_template
			| llm.with_structured_output(structuredAnswer, strict=True)
)
final_answer = rag_chain.invoke("How are the TINs structured for this country?")

In [ ]:
df = pd.DataFrame([final_answer.model_dump()])
#same as
	#df = pd.DataFrame({
	#    "answer": [final_answer.answer],
	#    "sources": [final_answer.sources],
	#    "regex": [final_answer.regex]
	#})
display(df)
